# Between-Comment Analysis Within Articles

For each article, we compute a **subcategory proportion vector** for every individual comment,
then measure how much each comment diverges from the article's mean vector using **cosine similarity**.

This captures within-article heterogeneity at the commenter level: do commenters on the same
visualization tend to focus on the same mix of categories, or do they diverge significantly?

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
alt.data_transformers.enable("vegafusion")

# ── Load all ace_classifications ────────────────────────────────────────
CLASSIFICATIONS_DIR = Path("..") / "ace_classifications"

rows = []
for p in sorted(CLASSIFICATIONS_DIR.glob("*.json")):
    with p.open() as f:
        rows.extend(json.load(f))

df_raw = pd.DataFrame(rows)
df_raw["article_id"] = df_raw["article_id"].astype(str)
df_raw["comment_id"] = df_raw["comment_id"].astype(int)

_TAG_CLEANUP = {
    "L3: Trend and pattern analysis": "Visual Observation: Cross-point Pattern Recognition",
    "L1: Elemental and encoded properties": "Visual Observation: Chart Structure & Text",
    "L2: Statistical concepts and relations": "Visual Observation: Data Point Extraction",
    "VO1: Chart Structure, Layout & Text": "Visual Observation: Chart Structure & Text",
    "VO2: Data Point Reading": "Visual Observation: Data Point Extraction",
    "VO3: Comparisons, Trends & Patterns": "Visual Observation: Cross-point Pattern Recognition",
    "Background knowledge": "Prior Knowledge: Background",
    "Personal/episodic retrieval": "Prior Knowledge: Personal / Episodic",
    "Evaluative / affective judgment": "Evaluative: Reactive",
    "Explanatory inference": "Inference: Explanatory",
    "Predictive / counterfactual inference": "Inference: Predictive / Hypothetical",
    "Information need / curiosity": "Curiosity",
    "Meta /Paratext": "Meta / Paratext",
    "Meta / paratext": "Meta / Paratext",
}
df_raw["comment_tag"] = df_raw["comment_tag"].replace(_TAG_CLEANUP)

EXCLUDED = {"Meta / Paratext", "Uncategorizable"}
n_before = len(df_raw)
df = df_raw[~df_raw["comment_tag"].isin(EXCLUDED)].copy()
print(f"Filtered {n_before - len(df):,} Meta/Uncategorizable → {len(df):,} sentences retained")

SUBCATEGORIES = [
    "Visual Observation: Chart Structure & Text",
    "Visual Observation: Data Point Extraction",
    "Visual Observation: Cross-point Pattern Recognition",
    "Prior Knowledge: Background",
    "Prior Knowledge: Personal / Episodic",
    "Evaluative: Prescriptive",
    "Evaluative: Reactive",
    "Inference: Explanatory",
    "Inference: Predictive / Hypothetical",
    "Curiosity",
]

SHORT = {
    "Visual Observation: Chart Structure & Text": "VO1",
    "Visual Observation: Data Point Extraction": "VO2",
    "Visual Observation: Cross-point Pattern Recognition": "VO3",
    "Prior Knowledge: Background": "Background",
    "Prior Knowledge: Personal / Episodic": "Personal",
    "Evaluative: Prescriptive": "Prescriptive",
    "Evaluative: Reactive": "Reactive",
    "Inference: Explanatory": "Explanatory",
    "Inference: Predictive / Hypothetical": "Predictive",
    "Curiosity": "Curiosity",
}
SHORT_LABELS = list(SHORT.values())

df = df[df["comment_tag"].isin(SUBCATEGORIES)].copy()
df["tag_short"] = df["comment_tag"].map(SHORT)

print(f"Loaded {len(df):,} sentences across {df['article_id'].nunique()} articles, "
      f"{df.groupby('article_id')['comment_id'].nunique().sum()} total comments")

Filtered 82,078 Meta/Uncategorizable → 447,985 sentences retained
Loaded 447,984 sentences across 191 articles, 52457 total comments


## Per-Comment Proportion Vectors & Cosine Similarity to Article Mean

For every comment we build a 10-dimensional proportion vector (one entry per subcategory).
Within each article we then compute the **mean comment vector** and measure each comment's
**cosine similarity** to that mean. Low similarity = the comment's category mix is unusual
relative to other commenters on the same visualization.

In [2]:
from numpy.linalg import norm

# ── Build per-comment proportion vectors ────────────────────────────────
comment_counts = (
    df.groupby(["article_id", "comment_id", "tag_short"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=SHORT_LABELS, fill_value=0)
)
comment_props = comment_counts.div(comment_counts.sum(axis=1), axis=0)

print(f"Proportion vectors: {len(comment_props):,} comments")
print(f"Subcategory columns: {list(comment_props.columns)}")
comment_props.head()

Proportion vectors: 52,457 comments
Subcategory columns: ['VO1', 'VO2', 'VO3', 'Background', 'Personal', 'Prescriptive', 'Reactive', 'Explanatory', 'Predictive', 'Curiosity']


tag_short                   VO1       VO2       VO3  Background  Personal  \
article_id comment_id                                                       
1          1           0.142857  0.428571  0.142857    0.000000  0.142857   
           2           0.000000  0.000000  0.411765    0.529412  0.000000   
           3           0.000000  0.000000  0.363636    0.181818  0.181818   
           4           0.000000  0.000000  0.250000    0.000000  0.500000   
           5           0.000000  0.000000  0.200000    0.400000  0.100000   

tag_short              Prescriptive  Reactive  Explanatory  Predictive  \
article_id comment_id                                                    
1          1                    0.0       0.0          0.0    0.000000   
           2                    0.0       0.0          0.0    0.000000   
           3                    0.0       0.0          0.0    0.090909   
           4                    0.0       0.0          0.0    0.000000   
           5                    0.0       0.0          0.0    0.100000   

tag_short              Curiosity  
article_id comment_id             
1          1            0.142857  
           2            0.058824  
           3            0.181818  
           4            0.250000  
           5            0.200000

In [3]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    d = norm(a) * norm(b)
    if d == 0:
        return np.nan
    return float(np.dot(a, b) / d)


# ── Per-article mean vector & cosine similarity ─────────────────────────
records = []
article_means = {}

for article_id, grp in comment_props.groupby(level="article_id"):
    vectors = grp.values  # (n_comments, 10)
    mean_vec = vectors.mean(axis=0)
    article_means[article_id] = mean_vec

    for (_, comment_id), vec in zip(grp.index, vectors):
        cs = cosine_similarity(vec, mean_vec)
        n_sentences = int(comment_counts.loc[(article_id, comment_id)].sum())
        records.append({
            "article_id": article_id,
            "comment_id": comment_id,
            "n_sentences": n_sentences,
            "cosine_sim": cs,
        })

df_cos = pd.DataFrame(records)

print(f"Computed cosine similarity for {len(df_cos):,} comments across "
      f"{df_cos['article_id'].nunique()} articles")
print()
print("── Cosine similarity summary ──")
print(df_cos["cosine_sim"].describe().round(4).to_string())
print()

# Per-article summary
article_summary = (
    df_cos.groupby("article_id")["cosine_sim"]
    .agg(["count", "mean", "std", "min", "max"])
    .rename(columns={"count": "n_comments"})
    .sort_values("mean", ascending=True)
)
print("── Per-article cosine similarity (sorted by mean, ascending) ──")
article_summary.round(4)

Computed cosine similarity for 52,457 comments across 191 articles

── Cosine similarity summary ──
count    52457.0000
mean         0.7163
std          0.1756
min          0.0154
25%          0.6295
50%          0.7597
75%          0.8447
max          0.9867

── Per-article cosine similarity (sorted by mean, ascending) ──


,n_comments,mean,std,min,max
article_id,,,,,
23,137,0.5482,0.1671,0.0513,0.9248
25,115,0.5495,0.1416,0.1909,0.8228
35,135,0.6021,0.2204,0.0976,0.9135
70,193,0.6080,0.1466,0.1548,0.8965
12,110,0.6124,0.2103,0.0915,0.9468
...,...,...,...,...,...
170,250,0.7868,0.1344,0.0645,0.9777
205,188,0.7882,0.1570,0.0972,0.9831
74,422,0.7926,0.1362,0.0417,0.9679


## Visualizations

In [4]:
# ── Strip plot: per-comment cosine similarity, grouped by article ───────
article_order = article_summary.index.tolist()

strip = alt.Chart(df_cos, title="Comment cosine similarity to article mean vector").mark_circle(
    size=20, opacity=0.5
).encode(
    x=alt.X("article_id:N", sort=article_order, title="Article (sorted by mean similarity)"),
    y=alt.Y("cosine_sim:Q", title="Cosine similarity to article mean", scale=alt.Scale(domain=[0, 1])),
    tooltip=["article_id", "comment_id", alt.Tooltip("cosine_sim:Q", format=".3f"), "n_sentences"],
)

mean_line = alt.Chart(article_summary.reset_index()).mark_tick(
    color="red", thickness=2, size=15
).encode(
    x=alt.X("article_id:N", sort=article_order),
    y=alt.Y("mean:Q"),
)

(strip + mean_line).properties(width=800, height=350)

alt.LayerChart(...)

In [5]:
# ── Histogram of all cosine similarities ────────────────────────────────
alt.Chart(df_cos, title="Distribution of comment-to-article-mean cosine similarity").mark_bar(
    opacity=0.8
).encode(
    x=alt.X("cosine_sim:Q", bin=alt.Bin(maxbins=40), title="Cosine similarity"),
    y=alt.Y("count()", title="Number of comments"),
).properties(width=600, height=300)

alt.Chart(...)

## Relationship Between Comment Length and Divergence

Do shorter comments (fewer sentences) tend to diverge more from the article mean?
Short comments may be dominated by a single category, producing extreme proportion vectors.

In [9]:
# alt.Chart(

#     df_cos,
#     title="Comment length vs. cosine similarity to article mean"
# ).mark_circle(size=30, opacity=0.4).encode(
#     x=alt.X("n_sentences:Q", title="Sentences in comment", scale=alt.Scale(type="log")),
#     y=alt.Y("cosine_sim:Q", title="Cosine similarity", scale=alt.Scale(domain=[0, 1])),
#     color=alt.Color("article_id:N", legend=None),
#     tooltip=["article_id", "comment_id", "n_sentences", alt.Tooltip("cosine_sim:Q", format=".3f")],
# ).properties(width=600, height=400)

## Most and Least Divergent Comments

Inspect the comments with lowest and highest cosine similarity to their article's mean.

In [8]:
# ── Merge back original comment text for inspection ─────────────────────
comment_text = (
    df.groupby(["article_id", "comment_id"])["original_comment"]
    .apply(lambda s: " | ".join(s.unique()))
    .reset_index()
)
df_cos_text = df_cos.merge(comment_text, on=["article_id", "comment_id"], how="left")

print("── 10 most divergent comments (lowest cosine similarity) ──\n")
for _, row in df_cos_text.nsmallest(10, "cosine_sim").iterrows():
    text_preview = row["original_comment"][:120] + ("..." if len(row["original_comment"]) > 120 else "")
    print(f"  art={row['article_id']:>3s}  comment={row['comment_id']:>4d}  "
          f"cos={row['cosine_sim']:.3f}  n={row['n_sentences']}  {text_preview}")

print("\n── 10 most typical comments (highest cosine similarity) ──\n")
for _, row in df_cos_text.nlargest(10, "cosine_sim").iterrows():
    text_preview = row["original_comment"][:120] + ("..." if len(row["original_comment"]) > 120 else "")
    print(f"  art={row['article_id']:>3s}  comment={row['comment_id']:>4d}  "
          f"cos={row['cosine_sim']:.3f}  n={row['n_sentences']}  {text_preview}")

── 10 most divergent comments (lowest cosine similarity) ──

  art=  2  comment= 373  cos=0.015  n=4  There is a storm problem. | We deal with this storm problem. | We should go down to Texas. | We should help those in nee...
  art= 40  comment= 577  cos=0.021  n=3  Sharon Hessney last lived through a hurricane in 1958. | Tonya Adkins is of Charlotte, North Carolina. | Tonya Adkins ha...
  art=129  comment= 264  cos=0.021  n=1  A human should have daily things.
  art= 64  comment= 308  cos=0.023  n=6  We have been moderating for 5 hours. | Gabrielle and Ethan are from Akron, Ohio. | Gabriel is of Polaris Expeditionary L...
  art= 41  comment= 173  cos=0.023  n=1  Serena Williams is the GOAT.
  art= 41  comment= 348  cos=0.023  n=1  Serena Williams is the GOAT.
  art= 41  comment= 548  cos=0.023  n=1  Serena Williams is the GOAT.
  art= 41  comment= 723  cos=0.023  n=1  Serena Williams is the GOAT.
  art= 41  comment= 948  cos=0.023  n=1  Serena Williams is the GOAT.
  art= 41  comment=

In [10]:
# ── Calculate average "order" (sentence or comment sequence) of each subcategory in each article ──

import glob

# --- Load all ace_classifications files into a DataFrame ---
all_classifications = []
for fp in glob.glob("../ace_classifications/*.json"):
    all_classifications.extend(pd.read_json(fp))

df_class = pd.DataFrame(all_classifications)

# Try to ensure "order" column is present (from ace_comments), otherwise will need to load ace_comments
# We'll merge in the order (position within comment) for each sentence from ace_comments

# Load ace_comments (contains order)
comments = []
for fp in glob.glob("../ace_comments/*.json"):
    comments.extend(pd.read_json(fp))
df_comments = pd.DataFrame(comments)
df_comments["article_id"] = df_comments["article_id"].astype(str)
df_comments["comment_id"] = df_comments["comment_id"].astype(int)  # to ensure merge works if needed

# Merge to get the "order" for each classified statement
df_class["article_id"] = df_class["article_id"].astype(str)
df_class["comment_id"] = df_class["comment_id"].astype(int)
df_class["order"] = df_class["order"] if "order" in df_class else None  # fallback, will override next

# Align on article, comment, and original_comment to get "order"
df_merged = pd.merge(
    df_class,
    df_comments[["article_id", "comment_id", "original_comment", "order"]],
    on=["article_id", "comment_id", "original_comment"],
    how="left"
)

# Clean/standardize category tags
_TAG_CLEANUP = {
    "L3: Trend and pattern analysis": "Visual Observation: Cross-point Pattern Recognition",
    "L1: Elemental and encoded properties": "Visual Observation: Chart Structure & Text",
    "L2: Statistical concepts and relations": "Visual Observation: Data Point Extraction",
    "VO1: Chart Structure, Layout & Text": "Visual Observation: Chart Structure & Text",
    "VO2: Data Point Reading": "Visual Observation: Data Point Extraction",
    "VO3: Comparisons, Trends & Patterns": "Visual Observation: Cross-point Pattern Recognition",
    "Background knowledge": "Prior Knowledge: Background",
    "Personal/episodic retrieval": "Prior Knowledge: Personal / Episodic",
    "Evaluative / affective judgment": "Evaluative: Reactive",
    "Explanatory inference": "Inference: Explanatory",
    "Predictive / counterfactual inference": "Inference: Predictive / Hypothetical",
    "Information need / curiosity": "Curiosity",
    "Meta /Paratext": "Meta / Paratext",
    "Meta / paratext": "Meta / Paratext",
}
df_merged["comment_tag"] = df_merged["comment_tag"].replace(_TAG_CLEANUP)

# Remove excluded categories, if any
EXCLUDED = {"Meta / Paratext", "Uncategorizable"}
df_merged = df_merged[~df_merged["comment_tag"].isin(EXCLUDED)]

# For each article_id and each subcategory, compute the mean 'order' (position within comment) for statements of that category
avg_order_by_article = (
    df_merged.groupby(["article_id", "comment_tag"])["order"]
    .mean()
    .unstack()
)

# Show the result (average "order" of appearance for each subcategory, per article)
with pd.option_context("display.max_columns", None, "display.width", 140):
    display(avg_order_by_article)

KeyError: 'article_id'